Cellule 1: Libs Importation

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

Cellule 2 : Loading and inspection of dataset

In [2]:
df = pd.read_csv("dataset.csv", sep=";")
print(df.shape)
df.info()
df.head()

(12070, 26)
<class 'pandas.DataFrame'>
RangeIndex: 12070 entries, 0 to 12069
Data columns (total 26 columns):
 #   Column                                                                         Non-Null Count  Dtype
---  ------                                                                         --------------  -----
 0   Date                                                                           12010 non-null  str  
 1   Service                                                                        11830 non-null  str  
 2   Departure station                                                              12011 non-null  str  
 3   Arrival station                                                                12011 non-null  str  
 4   Average journey time                                                           11830 non-null  str  
 5   Number of scheduled trains                                                     11830 non-null  str  
 6   Number of cancelled trains           

,Date,Service,Departure station,Arrival station,Average journey time,Number of scheduled trains,Number of cancelled trains,Cancellation comments,Number of trains delayed at departure,Average delay of late trains at departure,...,Number of trains delayed > 15min,Average delay of trains > 15min (if competing with flights),Number of trains delayed > 30min,Number of trains delayed > 60min,Pct delay due to external causes,Pct delay due to infrastructure,Pct delay due to traffic management,Pct delay due to rolling stock,Pct delay due to station management and equipment reuse,"Pct delay due to passenger handling (crowding, disabled persons, connections)"
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.0,870,5.0,NaN,289.0,11.24780854,...,110.0,6.51,44.0,8.0,36.13445378,31.09243697,10.92436975,15.96638655,"5,04",0.840336134
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56.0,406.0,1.0,NaN,213.0,8.479968701,...,32.0,5.363539095,9.0,4.0,20.0,35.0,16.66666667,16.66666667,8.333333333,3.333333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.0,226.0,0.0,NaN,21.0,6.23968254,...,11.0,2.938053097,6.0,1.0,22.22222222,27.77777778,16.66666667,16.66666667,5.555555556,11.11111111
3,2018-01,National,PARIS MONTPARNASSE,NANTES,216.21,508.0,3.0,NaN,71.0,7.235211268,...,39.0,5.292211221,18.0,NaN,33.33333333,22.22222222,16.66666667,20.37037037,5.555555556,1.851851852
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94.0,472.0,4.0,NaN,224.0,6.784672619,...,42.0,4.882371795,10.0,0.0,15.78947368,45.61403509,NaN,15.78947368,1.754385965,1.754385965


Cellule 3 : Reperage of null values and delection of double cells

In [3]:
print(df.isnull().sum())
print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.shape)

Date                                                                                60
Service                                                                            240
Departure station                                                                   59
Arrival station                                                                     59
Average journey time                                                               240
Number of scheduled trains                                                         240
Number of cancelled trains                                                         239
Cancellation comments                                                            11493
Number of trains delayed at departure                                              240
Average delay of late trains at departure                                          239
Average delay of all trains at departure                                           241
Departure delay comments                   

Cellule 4 : convertion of date number columns to correct format

In [4]:

df["Date"] = pd.to_datetime(df["Date"], format="mixed").dt.to_period("M")
df.head()
print(df.dtypes)
num_cols = df.columns.drop(["Date", "Service", "Departure station", "Arrival station"])
for col in num_cols:
    df[col] = df[col].str.replace(",", ".").str.strip()
    df[col] = pd.to_numeric(df[col], errors="coerce")
print(df.dtypes)

Date                                                                             period[M]
Service                                                                                str
Departure station                                                                      str
Arrival station                                                                        str
Average journey time                                                                   str
Number of scheduled trains                                                             str
Number of cancelled trains                                                             str
Cancellation comments                                                                  str
Number of trains delayed at departure                                                  str
Average delay of late trains at departure                                              str
Average delay of all trains at departure                                               str

Cellule 5: replacing of missing values(Not A Number) by the median of columns

In [5]:
for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())
print(df.isnull().sum())

Date                                                                                59
Service                                                                            240
Departure station                                                                   59
Arrival station                                                                     59
Average journey time                                                                 0
Number of scheduled trains                                                           0
Number of cancelled trains                                                           0
Cancellation comments                                                            11896
Number of trains delayed at departure                                                0
Average delay of late trains at departure                                            0
Average delay of all trains at departure                                             0
Departure delay comments                   

Cellule 6 : Delection of not utils columns(comments columns)

In [6]:
df = df.drop(columns=["Cancellation comments", "Departure delay comments", "Arrival delay comments"])
print(df.isnull().sum())
print(df.shape)
df = df.dropna(subset=["Date", "Service", "Departure station", "Arrival station"])
print(df.isnull().sum())
print(df.shape)

Date                                                                              59
Service                                                                          240
Departure station                                                                 59
Arrival station                                                                   59
Average journey time                                                               0
Number of scheduled trains                                                         0
Number of cancelled trains                                                         0
Number of trains delayed at departure                                              0
Average delay of late trains at departure                                          0
Average delay of all trains at departure                                           0
Number of trains delayed at arrival                                                0
Average delay of late trains at arrival                          

Cellule 7: Feature engineering

In [ ]:
df["Year"] = df["Date"].dt.year

df["Month"] = df["Date"].dt.month

#Categorizing average delay of all trains at arrival

def categorize_delay(delay):
    if delay < 5:
        return "minor delay"
    if delay < 15:
        return "average delay"
    if delay < 30:
        return "major delay"
    return  "severe delay"

df["Delay categories"] = df["Average delay of all trains at arrival"].apply(categorize_delay)

df.info()
df.head()

<class 'pandas.DataFrame'>
Index: 11487 entries, 0 to 12060
Data columns (total 25 columns):
 #   Column                                                                         Non-Null Count  Dtype    
---  ------                                                                         --------------  -----    
 0   Date                                                                           11487 non-null  period[M]
 1   Service                                                                        11487 non-null  str      
 2   Departure station                                                              11487 non-null  str      
 3   Arrival station                                                                11487 non-null  str      
 4   Average journey time                                                           11487 non-null  float64  
 5   Number of scheduled trains                                                     11487 non-null  float64  
 6   Number of cancelled tr

,Date,Service,Departure station,Arrival station,Average journey time,Number of scheduled trains,Number of cancelled trains,Number of trains delayed at departure,Average delay of late trains at departure,Average delay of all trains at departure,...,Number of trains delayed > 30min,Number of trains delayed > 60min,Pct delay due to external causes,Pct delay due to infrastructure,Pct delay due to traffic management,Pct delay due to rolling stock,Pct delay due to station management and equipment reuse,"Pct delay due to passenger handling (crowding, disabled persons, connections)",Year,Month
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.00,870.0,5.0,289.0,11.247809,3.693179,...,44.0,8.0,36.134454,31.092437,10.924370,15.966387,5.040000,0.840336,2018,1
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56.00,406.0,1.0,213.0,8.479969,4.567119,...,9.0,4.0,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333,2018,1
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.00,226.0,0.0,21.0,6.239683,0.286283,...,6.0,1.0,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111,2018,1
3,2018-01,National,PARIS MONTPARNASSE,NANTES,216.21,508.0,3.0,71.0,7.235211,0.980000,...,18.0,3.0,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852,2018,1
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94.00,472.0,4.0,224.0,6.784673,3.229701,...,10.0,0.0,15.789474,45.614035,18.750000,15.789474,1.754386,1.754386,2018,1


Cellule 8 : Creation of "cleaned_dataset.csv"

In [8]:
df.to_csv("cleaned_dataset.csv", index=False, sep=";")